In [1]:
!pip install firebase-admin

   ---------------------------------------- 0.0/13.2 MB ? eta -:--:--
   - -------------------------------------- 0.5/13.2 MB 2.8 MB/s eta 0:00:05
   -- ------------------------------------- 0.8/13.2 MB 2.6 MB/s eta 0:00:05
   --- ------------------------------------ 1.3/13.2 MB 2.1 MB/s eta 0:00:06
   ---- ----------------------------------- 1.6/13.2 MB 2.1 MB/s eta 0:00:06
   ------ --------------------------------- 2.1/13.2 MB 2.0 MB/s eta 0:00:06
   ------- -------------------------------- 2.4/13.2 MB 1.9 MB/s eta 0:00:06
   ------- -------------------------------- 2.6/13.2 MB 1.8 MB/s eta 0:00:06
   -------- ------------------------------- 2.9/13.2 MB 1.7 MB/s eta 0:00:06
   --------- ------------------------------ 3.1/13.2 MB 1.7 MB/s eta 0:00:06
   ---------- ----------------------------- 3.4/13.2 MB 1.7 MB/s eta 0:00:06
   ----------- ---------------------------- 3.7/13.2 MB 1.6 MB/s eta 0:00:06
   ------------ --------------------------- 4.2/13.2 MB 1.7 MB/s eta 0:00:06
   ---


[notice] A new release of pip is available: 25.0 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import firebase_admin
from firebase_admin import credentials, db
import pandas as pd
import datetime
import requests 
import pytz
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from tabulate import tabulate
import time

# --------------- Firebase Initialization ---------------
if not firebase_admin._apps:
    cred = credentials.Certificate("E:\\project2\\serviceAccountKey.json")  # Adjust path if needed
    firebase_admin.initialize_app(cred, {
        'databaseURL': 'https://my-water-flow-project-default-rtdb.firebaseio.com/'
    })

firebase_ref = db.reference('/FloodPrediction')

# --------------- OpenWeatherMap Setup ------------------
API_KEY = "2c2f3bfca00102f597b626b3d6a6e929"
cities = ["Papanasam", "Aryankavu", "Thenmala", "Kalakkad", "Thenkasi"]

# --------------- Load Dataset and Train Model ----------
df = pd.read_csv('flood_data_rainfall.csv')

features = ["Papanasam", "Aryankavu", "Thenmala", "Kalakkad", "Thenkasi", "Total Rainfall (mm)"]
target = "Flood Risk"

le = LabelEncoder()
df[target] = le.fit_transform(df[target])

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# --------------- Rainfall Data Function ----------------
def get_weather_data(city):
    url = f"http://api.openweathermap.org/data/2.5/forecast?q={city}&units=metric&cnt=2&appid={API_KEY}"
    response = requests.get(url)
    data = response.json()

    if response.status_code != 200 or 'list' not in data:
        print(f"Error fetching data for {city}. Response: {data}")
        return 0

    rainfall = sum([forecast.get('rain', {}).get('3h', 0) for forecast in data['list']])
    return rainfall

def get_all_rainfall_data():
    rainfall_data = {}
    for city in cities:
        rainfall_data[city] = get_weather_data(city)
    return rainfall_data

# --------------- Prediction Function -------------------
def predict_flood_risk(rainfall_data):
    X_new = pd.DataFrame([rainfall_data])
    prediction = model.predict(X_new)
    prediction_label = le.inverse_transform(prediction)[0]
    return prediction_label

# --------------- Dashboard Display ---------------------
def display_dashboard(rainfall_data):
    total_rainfall = rainfall_data["Total Rainfall (mm)"]
    table_data = [[city, f"{rain:.2f} mm"] for city, rain in rainfall_data.items()]
    headers = ["City", "Rainfall Rate"]
    print("\n🌧 Rainfall Rates from 5 Cities:\n")
    print(tabulate(table_data, headers=headers, tablefmt="pretty"))
    print(f"\n💧 Total Rainfall: {total_rainfall:.2f} mm")

# --------------- Main Update Loop ----------------------
def update_firebase():
    while True:
        rainfall_data = get_all_rainfall_data()
        total_rainfall = sum(rainfall_data.values())
        rainfall_data["Total Rainfall (mm)"] = total_rainfall

        prediction_label = predict_flood_risk(rainfall_data)

        note = "Potential Risk and Caution Tourists at Coutrallam, due to present weather condition"
        action = "Issue Flood Alert and Caution Tourists" if prediction_label == "Yes" else \
                 "Monitor Closely – Potential Risk and Caution Tourists" if prediction_label == "Unpredictable" else \
                 "No Immediate Action Required"

        india_time = datetime.datetime.now(pytz.timezone("Asia/Kolkata")).strftime("%Y-%m-%d %H:%M:%S")

        firebase_data = {
            "FallsName": "Coutralam Falls",
            "PredictedDate": (datetime.datetime.utcnow() + datetime.timedelta(days=1)).strftime("%Y-%m-%d"),
            "PredictedFloodRisk": prediction_label,
            "RecommendedAction": action,
            "Note": note,
            "Time": india_time
        }

        firebase_ref.set(firebase_data)

        print("\n✅ Updated Data Sent to Firebase:")
        print(tabulate([firebase_data], headers="keys", tablefmt="pretty"))

        # Display Rainfall Dashboard in Notebook
        display_dashboard(rainfall_data)

        # ⏱ Wait 60 seconds before next update
        time.sleep(60)

# --------------- Start the Prediction Process ----------
update_firebase()

FileNotFoundError: [Errno 2] No such file or directory: 'E:\\project2\\serviceAccountKey.json'

In [3]:
!pip install tabulate


[notice] A new release of pip is available: 25.0 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
